# Gemma 4 Developer Agent — Phase 0 checks, EXP-000 gold control, baseline eval (4x L4)

One notebook, three switches (`RUN_PHASE0_CHECKS`, `RUN_GOLD_CONTROL`, `RUN_AGENT`). It mirrors the
organizers' Getting-Started recipe (wheelhouse install, `VllmServer`, `swegemma.evaluate.Evaluator` with the
subprocess sandbox) and writes everything under `/kaggle/working/runs/<EXP_ID>/` in the layout that
`tools/analyze_traces.py` and `tools/estimate_budget.py` read.

**Attach as inputs** (Add Input): competition `gemma-4-developer-agent`, dataset
`metric/gemma-4-developer-agent-wheelhouse`, model `google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2`
(or the `.../gemma-4-e4b-it-qat-w4a16-ct/2` / `.../gemma-4-e2b-it-qat-w4a16-ct/2` input when the zip's agent.yaml names a
small variant; the vLLM cell takes the model from the zip).
**Accelerator**: NVIDIA L4 x4 (competition-attached notebooks only). **Internet**: off.
The submission zip comes either from a private Kaggle dataset you upload (`SUBMISSION_ZIP`) or embedded as
base64 by `python tools/build_notebook.py --embed dist/submission.zip`.

**Status: UNTESTED on Kaggle.** This file was written without access to kaggle.com, the wheelhouse or the
harness source. Cells marked `# VERIFY` use API names taken from a community mirror of the starter notebook
and may need adjusting against the real `getting-started-gemma-4-developer-agent` notebook. Read the
starter notebook first and diff the server/evaluator cells.

Edit the parameters in the next cell, save a notebook *version* per experiment, then download
`/kaggle/working/runs/<EXP_ID>.zip` and run `python tools/analyze_traces.py runs/<EXP_ID>`.

In [ ]:
# ---------------------------------------------------------------- parameters (edit me)
EXP_ID = "EXP-064-loc"   # overridden by --set
RUN_PHASE0_CHECKS = True          # Q1/Q2/Q3/Q7/Q9 answers + HARNESS_README copy + sample_submission dump
RUN_GOLD_CONTROL = False   # overridden by --set
RUN_AGENT = True                  # run our submission on the sound tasks
MEASURE_DECODE_SPEED = True       # Q10: one timed completion against the server

# Task selection: explicit ids win; otherwise the first N_TASKS of the (optionally repo-filtered) dataset.
TASK_IDS = ["fastapi_14301", "fastapi_14448", "fastapi_14605", "httpx_3672", "requests_7205", "requests_7309", "rich_3006", "rich_3468"]   # overridden by --set
N_TASKS = 8   # overridden by --set
REPO_FILTER = ""                  # e.g. "Textualize/rich"; "" = all repos
EXCLUDE_IDS = []                  # known-unsound tasks from a previous gold control (experiments/README.md Q5)

# Where the submission comes from (first that exists wins). Upload dist/submission.zip as a private dataset,
# or embed it: python tools/build_notebook.py --embed dist/submission.zip
SUBMISSION_ZIP = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-sub-v1/submission.zip"   # VERIFY path
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVxWNoSemwEAAN0CAAAKAAAAYWdlbnQueWFtbH2STW8TMRCG7/4Vg3ogleJ8ESK6ByQOlXoACQkOSBWKJvZs1sVf9XhZwq/H620akIDL7kjz8b7zjK/g9stHudqtYbPa7OR6LVdvGkghZPhEjz35bNC+O5Y/yLfw3ropHrrABMHbEzyEAxiGHKBDryG07RjnjqbJrzfweQPRRLLGE8x8GID7wx7HObw8JxYndPZ6DhwgBs7S+O9BYTbBgwouoqphhwzoLzaqTxc0WZid1f4o/4lJzyFRS4m8ouUluXD6eiGu4EPtHp0zulisHAETVf+JUFN6yTA7knMot5K2B2myfMQshy2ud1LleRH0rTny8tw/rbIoFIJluP860qEfJaVMfrL56qYZ9+i9JmWLnK7FYA1n0AkHhgGTL7O4OnN4Ak0t9jYXzx4dNcADTQxF/e7LHObmGY3QxCqZOK7awF2ZwnWnjPztfJ9xv/mo1+YqgzGWez5fRFSwDfxneWE859SrSeWF8cr2miCmgrkcdzxP4SwqiaagEMUaJcy0L9BytV3h/db7V5ri8mIaASCfqvYRc9f88zmJX1BLAwQUAAAACAAAACFc2PyhGNUAAAAYAQAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD2OMU/EMAyF9/wKSywgtVXauys0GwMbSIwwVbnGaaK2TpQ6Av49oScx2e/z89O7g5eP91peOuhk19dtW8tHBezwxk8n2PUWV08zWL9iBZkmp2lGU8HudEID15/Dn1AbTKDJHFLH8lX0faD1ZjBJW8a0g/HWYqrAU8nFMv7OPkH4Ivh8fnt9aATjFjFpzgkVtI0UHOIYFchmuBz7oqA/i01/jyFzzDxyWJB2BU/t0Al2npbSeZwCWT8rAfCPrtnMyArOcugL9zSt2eDILuTZcUnglFH8AlBLAwQUAAAACAAAACFc6A3n1sQBAAAZAwAAEAAAAGV2YWxfY29uZmlnLnlhbWxdUkuP2kAMvvMrLO1hF4mgJGxKCFIvqz23hx56QyYxMGpmJhpPYPff154Q+rgl39j+HvYTvP/8nuVVCWVefsmKIsu3DVj8OETv+0OLfc9Q1pB9hdcK0HXT2xgcQ50rXJT5HqKx5Md4YGq966TjNf9TLG8Ha9wYSXoAA02cm80zrxdP8KYcDcQLQSDsKDwz3HzgCJaQx0AdDMgMhqHYQOtHFwWalL2kSUImxNF4B7t1vUzMOg6HoTcUwBGJJjyKQtjAIEgX8BThhToTV9BeqP21gtF1fgnsoajg5AOczJWmQoahH4Udopd+6wPtNY2e8CqWgvc2NRx9vNyVt8jyghG8a+muclMu1e4Pza4BulL4lFkd9frtpPR0B/Gsv8moOOQbnclaFECMf8T18NmU221W7nbLNXxz9EhnMlirSrEwZQlpVSuoJiczkCKqtLLMHzHNpZUKTz72ul3xKTpQBnLEEOd1hNFpv5E3kmHGcRRGdXEk487QihZruuw+faXJ6pBMG/nib6LZpTXFgI+QZJUBDQuJXFeSl6dstY7RphNh7yRJumI/oi69WcD/B9joBQr87yE3srUZ/OsoG6hndNqNmF78BlBLAwQUAAAACAAAACFcat9W8yQHAABvDwAAEgAAAHByb21wdHMvYXBwbGllci5tZHVXwZLbNhK96yu6kkOyKUoTz4wP8SU18djZ1KaclOOqVE5DiGxKKJEAFwCt0W7tv+/rBkjKXu9lhgJAdPfrfq+bf/mJTGAyjvh55JDITMk7P/gpUvRdOssuu4N1zGFH7zn6/iNTOjJNkQPZGCems01HXYuD6XuOCe9OoWFqjsYd5LjBkkk2dpZjPplM4pb2fDQfrQ+7zeYDVo+2bdnRIZgWl5tx7OX8BZfRwSZqbddR8mSoCxyPFHj00SYfLoigpTC5SDZF8mdHCW7EHf2SyPFHXBZ5vimwid5Zd9jRA+IexnTJN8fG41r6FwePLcKvwM0Sy/URiSCZeCJc7ch2+eLeN6ZXw3Q2EXZa8kG2igsBMBvdR7hff00/Te2Bkwaulw0cozkUbIqzyft+i2t7jTDZgWmvr4mHiERxwk5nkZZzsIlDVEN77uCqXFLlV2Eln/8mklyIv35yCRaQ5LwfzdX1fx4RGiOzsNDbAfAfgW2lB88+nAAgpcCMGsgJa8kI/PidgW3xPGCdrMuOq714jRm1npxP4lhzpDjtYeZpNKk57uitdRY5NlKI3sl/Uy6FQVwZPfXAKSh2OJQsQEI1sRrLCH9gM2iVSGUDhDaYLsXNTws2gnVAIQKjGU0/suOMR+NbzpWlV0g+nW5o2Vf/C7yGoUb0LKjzbFBB6nUX/JCZkG/bERJvYW9K44Rt3/f+HOH3++xHPvVq8++xN+7H/2w2j3IvvcCKWnh6sS7eLou36+Ldsni3Lt4vi/fr4stl8aUsPoDbZt9zjgSYAnp6+8uvb1AHTsOm3359pD3q/aQYGHr35s/8G4w7OAHXlLcL9y+R3v1Gj+8f3n6oJIFKuwoZbE7x+v1KKCN4xUIVANyXbL7Y0euj9zHrT3NkOV7q3LiLckVLe/PIjUXqJAOAMfh2apJF7nB0zV/JLHRMqla2hKWX7PcrmI9HqZt6vKQj3t029NVut/uqziHxM4fGxiIGWejaXDGLHFonmV0NthybYPccNUrxTi6XIFWtbuTvk8Y7Xmh70tUnJ6Tc/rOYXS8roZTqHr0VbiUo17ssNhM0QNS7mKirxdiuVvtLYANNzibdai0UDq/XGfFbFCkQETuZOkI8H1Cemw/mNPeBtVZ0Xxb1DL2o6Laiu4ruK3oJRgvEwhBE/moj2RRdqyXbGnW9Iqe3QahkWahfywOE4QjXpTau6g+bvm+fYgpQpFqTKieWitITjs/ziQpJGi1ypdzsL7Iv/E2iSm61jkrucm7FywxzhFpB05LEEKgztkcqDx4qEow9HEVM9RXHz2m+5XGWuFSQVJXN5NB4c+3ELFDz4a2okR7P9X+7uKNiklsjZEvSrI1C6QBkGxy+W33XVRTyOdfp3G+vWSDqieqv6MQ8YkPYqTbgT0weS+EiYp8zvKOfvYQZOVPqfre5/8yc4pJjk44XMSjoUb+Indn7QozFIRHjwqIK9dj6K0/0qlomADUg7263qkjCiuCnA7aBxFPjhwFu5yo4+C+kY/Pyi856h0LYc2NEDXxHA7CRmDv7nKYghLWD8oxD8CHzd0SW+Nmi8eDgHvk6oXlLeYK30l618XtEIZPGPz7BtvoCsNWV01fgCg3vSjdG8eW6GQXUdrOVUERG4M0gKlhIWKoDrRzTj+zn85VuXEpVBdBd9lRW5jfMwYADGe+Vl7tsaTG/VJ9So5ZSLQSW+pubZOCDXR61pYmUtTpBOLDolAX6mv++TFI6wIlfrQcxzrPwLfPL678/vPs5tyPtLeLg74E72Lh2+1NlGKaYCnlnXs0agKy5pp9ayYZ1GEExgcF36TdlgL2ebUdTvENN9DywCq+OCw460FxzHryCRckPnrGFh8kF7nX4lWYhnmfBbhBd4rn9Ua3i/x06gSj3d086NuoPtDotKmGJBJx7x6rvOwxOReHhUTe/uHhlHYYe5G1E70XFKngPaJ/oTQqO9KgxoRJZ5by+ScP4f3g2Nxudf5YLBFl5FYVc38ioCMQaScdb3ei5S7PSXk3wYJ8fpFTBMp8ZqiO3DlkgqQ6ZwiWlBDTnDx0X4fxfOnwbpMZk2mim9zKDaIGKH9ezJcDRGfQsdbgqwWczezzZcWRpJgtyOUzc2ntVhzgBBrjuQ54/RGYEXZFuvzaPPKoL84RYol5mnVVkGEqaep2fJ9fp3Cu/ytSeZIjFKZF65Weh44KQ1B2Wl4lJ+mTuDJ9rZp25afIZtZAp/Ak8n1ip8THwJF8kU6xLI0ESpeDBgh++F6nyrhXkBrwEVKovXmkHUN8iyv6SE6iFAxOvvTKP1/FGZyjEa3rh+2X+DtvRGx3P9FMJn6OtxFu+zQYPIGyjpC0TKXRvm7/t8vcETP3BJkjnnlKU6fC6Nunb+maKQUh0g7LgPuqjH1P9N6Sf8U0AVWjka1S/l4trhUrc6vVyqHxawY198EaGjLH3IavJ5r9QSwMEFAAAAAgAAAAhXIOdG63WAgAAxgUAABQAAABwcm9tcHRzL2RyYWZ0ZXJfMS5tZJVUS08bMRC+76+YWy8QiVa9RIiKkvCQUCIhJJSjs+tkLbyerR8sq6r/vd/YmwBtL70QvJ7X9xhvOJHymgZvovZ0Rryjr2Rco3uNPy5ON2FGG06elNtbPafYagqdslaHSD2HYLZWU93iWs/oKv+SCrTTA1njdJDDYKylhk9IxVxAv6oa2VbVOp+vbi9XN8scL2OM0s9yraJhhybGxUBR0l2D31S35Di2xu1J26BnVbX5kGEC3a+vLh/v1quCS3p43bOPtNWWhxktvNrF0kg3JhI7O0qYB8QUNBl0vL67X+aWcnhY3qBcGcHsHYM5/aL9SCxZx94zuvvQrgV6x+/HGUxsSR3rZZILq0YYLiSs1rR4uLx+LIhb7Wi5WhScKPmipWZktiEHdAlagBGqoQu+jDP6LjCFCIVJVKP9p3AYCQQpCryLg8hvQkiTrPnfgjD1KE87I51StwUtzRFiOCGtIMEEZGcw9DTmJGydvBcDRf0ahX7hz+u9SIOqk1eq6icM4L79qqqnzAG7Yz7KmUbhW9YmtrCNwtR9bw24zm7qVcA9nMFyVeJACCVX6jdo8JDg0nl1Suv7hXBRc28ABAEebVBpx/79yXOXURRt/nLiSYEsaEzekIPisRBds4tmnzgF2iLnWQp8EcRnn8sqFJrLVqDyxAT2ABmCHtPEgd8gTKGBSRxag9ZQjMquxrKtxIAYpbapgQTFOZksE7IdjCvHgmcGIlbLJ7kUPjKWg+3Jad0UatAYCudUeFJ1+j1YCnG0sufcj5P/Q/SyiEcDQekfyXjMncW0o/RdcB6oyISXA4/K9Ek16Mpdp2XH1ZZTeSEOJjn99z4dFPKYcCybir11z44H9787Vdw3ljcuDJnSw+yIA12wSZez/nh1Slvp0yvZhTBNKvtGe1lTwBWPfbBRBfbn1fn5eSXRbCeZq4uLiwr6vF25wxOar55uN/PskgCqNAxQYfzqN1BLAwQUAAAACAAAACFc90Lm3zgDAACcBgAAFAAAAHByb21wdHMvZHJhZnRlcl8yLm1klVRNa9tAEL3rV8ytF9fQll5MSEljpw0EB0Ih5LiWxvbS9Y66H1FE6X/vm125sUsvvQhJOzvz3ps38ySZTGAagk0c6D3Jlj6S9R33jIdP00mc05PkQMbvHC8o7ZlakyPP6QbBuECjHjtpTbLi30QKvMMLDXtGeo0fgvgdPRuXmSRMnxvem2eLTxtpa0NM1AfpcsvdDLU6aveoyGSTpgg8I35mT3ZbMl5/vVp/WZGznhX3GQLqOLbBbjiSQeYX2uagKaiTwccU2BzmdFvzTFDjXoZIXohNcBahEQlb1JxAaGjvTKtvJp1V9+bAcd40T2cQwOnu/vrq2+39uipbi/USEog7Gea0DGabKnLuwFK8GyvVBUFeEI90c3u3KmLox8PqC9JVcezOC8SFJGEkKeyOtU+4lXJ7A315QASwe6A5BsaZNuMEpkZCg2MdBfHn9B0aHRObrqQ/+X12idgqllm1TkERrfqmarW+p+XD1c23QgGHnlbrZdUOeZ5ZEyURF0vAIcMTXhLs5hz+jHP6rNKpuAbsTMehuK3QhOgGbdumQU1tY8yTWctrVS33SA9LaKV82EDq7lQNNu2eBjAovgHoCSbxi2kBI4egY5H4JWlLtSeTf5C1OgVkfsIo/tOvpnksGoj/cx/pbGfwr/S7OAkNMX1fLDdYsOwNRIbUSIijGgdBKPuav0OBh+w4Lpq3dH+3VC1a6S2XeQkog0xbtPXkK8ihsJha9Pe4zCplZWPL3B9dlKrQrfhkd1lypA3ufNcEH5Txu/elq7HKXF4183Fmeocbyh5o0iCvFKbQKDrb0kLWWM0vvsVWWaupAaV1uUMLqnOKWDYWO2DjlM/KZw4h1qtHPVQ9CpfjKJFn7vi4StDhchWexMyekqWYRofS19KP00xhT8C4rwZCp39kG4C7NNONWncpBVBtE8eEVTn9Mh2qyuGACtBwIznVxTmZ5O0/ZvR1iCBYNGOdfuwC/91jcf3vTFX3jXVzx6FIesSOOF26Eg7lFgDvlSy7qAauZcvKMzoLcUKq80Y7HVPQVY+d2aiB+ovm4uKi0WhxU5uby8vLBv15PfI8nBw9fn1aFJdESMUwQAP4zW9QSwMEFAAAAAgAAAAhXJGJ3+8KAwAANgYAABQAAABwcm9tcHRzL2RyYWZ0ZXJfMy5tZJVUTWvbQBC961fMrRfHkIZeTEhpY6cNBAdCIOS4lkbykvWuuh9RROl/75tdOVaglx4SrN2dmffevJlnl0h5psHryJ4uyLX0hbRtuGf8s3G6CUt6dsmTsp3hFcU9U5eUb5Z0K690O+YzZxnBfYrkPIWoIudjHUJiCmoM1DkOyOlst0CyhlSDP9p5Zeu9BNV7rl+oxS9+U3U0kldFqlXgBRlWr9p2pTxb9spQr+KeEqIBjZsS+cp+JIdXvqBZVlVGb1ytonYWgOju/vrb4+39tnCWjJ575yPt2LhhSWuv2kijhHGjQchmLOxBPwVwioFubu82mYV8PGx+IF1hpTvroOocyLE2FPtQbq8CtTzgBYhaoDk+DAvRYwZTXlr3XkdAvN+eg2iIrJqcfnb8IYhYC5ZFaWpGEbR0lIxG57b3tH74dvOYKeDS0ma7LtohzytLouicCfnBIYWIE+mNMTgZl/RdpBNxFdiphv2ncKQJ0RUF18ZB7JYNsTp5o6iWeqSnVkuldNhB6mauBitYZAADZGo1QE8wi1OoTt6LYSO/RWmp9MRzJ+1G1uIPkPndG2W//qmqp6yBOHaKRzrdiGVzv7Pt0BDV90ajO4M2YjaIDKmREFflHQQ5+Q8FHpLhsKrO6P5uLVrUrtcgggceZZCpzTZ///LukFlMLQLyce7VRaEsbHSeyKOLYhG6dhi/LrkUaIeYF0lwIYzPP+euhiJz/imZJyWoN4gQ9kATBzcbofI0OBLX15A1FPM7W/OStmJqQKlNatCC4pwslg7ZDtqWz8JnCSG2mye5FD0yl+MokWVuijQojA7nUFkUB56TxSYZDUpfu36cZipEf1wEZbl4/pW0B+5pbUjdtcuASps4RCyx6Ui2Tu0OB1SAhjuXYk51NMnZP2b0NESy2mSVyfRjF9gX6wb7vzNV3DeWnRqGLOls5UEu2OSQowB4L2TZBDFwKSt18uoDtIJU5o06GVPQFY99sFEF9VfV5eVllfe0mdpcXV1dVejP6cryMLt6+vm8yi4JkIphgArwq79QSwMEFAAAAAgAAAAhXEG7n3knAwAAeAYAABQAAABwcm9tcHRzL2RyYWZ0ZXJfNC5tZJVUTW/bOhC861fsrRfXQL8uRpCijdM2QOAAQYAgR1pay0QorkpSUYWi//3NkpbtoO/Si2Fxyd2Z2dl9koFMYBqDTRzoI8mOPpH1DfeMH58OkbikJxkCGd86XlHaM9XiUzB1WtAooaGdhPxnSQ/mmYlfOEwUp24rbkHRtt6kIfCCOo7RtPgTE04i0jSMtA0FxgVPL8YNjAIm5So2RnwG/jnYwHGRb3ZaoEDA2z5IM9So+Atg3ISARIT2RsFl5Fd7rp+JTb2fEzWoHqxvybTG+phoUnKb60dy1nOkLYMO6ykKxpHDsqoyfye1SVY8cNHt3dWXh5u7TVFN8QTuJSS8djIuaR3M7pCZG5tIfEbHAQIOgGhTpG83t9eZk37cX39HusIRgimCIqPoq2PtJd28Krc3kXYMkKqaB5r5IuRCU85g6k0vxzoK4hh9RyoEmyanPzt+9YjYKpZFsUVGEa16IgtHmzta33/59pApIOjperMu2iHPC2uiJOJi6eMA5b0kqo1zOJmW9FWlU3EN2JmGw5s404TohqLs0qiGzb5YnSxSVBt6pKed1UpDt2Xt9Jka2QIjGCDTzjqeYRbvUD2EoJZP/CtpS7UngVttN7LCUb5lkPndO+M//6mqx6yB+ON7pLONwVnud/YwGmL63ll0Z7Rg2RuIDKmREKFyD4LQ4Ev+BgXuB8dxVb2lu9u1alFLb0HkaOk8a2dfQbrM4tAiIJ/OvboolJWNzTM9uygVoXWQbTsIhnGLN8+a4IMyfve+jEORuUyGjkRBSr3DC2UPNGmUE4XD1Sikrq8hayzmF1/zkjZqakCp3aCjX5yTxbIx28H68ln4LCGETiaCqkfmMo8SeeamSIPC6HB+Ck+ajs/JYtwnh9JX0k/zaioL4O8dMy8SrbuWDKi0iWPCMjkcmQZVpetQARpuZSjrajbJ2/+Z0dMQQbBopjL92AX+2cvo/3WmivumspV1Ryn50xKEXLBJl18B8F7Jsotq4FJW6/RGZyEekOq8UatjCrrqsVc2qqD+qrq4uKj0trhDm6vLy8sK/TmFPI9noccfT6vskgipGAaoAL/6D1BLAwQUAAAACAAAACFc+DbFOCwDAACVBgAAFAAAAHByb21wdHMvZHJhZnRlcl81Lm1klVRNa9tAEL3rV8ytF8eQll5MSEnjpA2kNoRAyHEtjewlq111P6KI0v/eN7uyY0MuvRiv5vO9eTPPLpHyTIPXkT19JdfiR9uGe8aPjZMlzOnZJU/Kbg0vSFGj25a9OISd6nlOt2J2cYcsUwgN2hjq1AsTPpPbvGqXAtU7JEHALzGo6QkPFWnLMWTfoDqmDe8UQjxtxpOC3qUoPdhSLUQVucudRJ/qmDzPDsY22TpqZwEpJ/a8xWtGyJp76nsX0CxtvLL1Dp8b9nO6i9SlIAkFge56UwoM0qTE6RCSJPudtOcwr6pMjnG1KsUC3a+vrx7v1qtCaSndOx+ByrhhTkuv2kijhHGjIzlrRnHzQJYCKoCJ27v7GyBp8uPh5gfSzcp7ax2Gxq/sx4nzfW00f1JupwK1PLDgVRbd7B1DJuGoTfG07lBHmjhYz8FfiKyanP7o80kQsZZeZkUAZZBaBENGW6bVmpYPV7ePGQKMlm5Wy8Id8ryyJIrOmZAd8gAwRaoVZqDsOKfvQp2Qq4BOYVKfwh4mSFcUXBsHUXMez+J9UoW11CM9tVoqpW4DqptjNlhBAAMQIFOr0fTUJvGbqtFG8ll9kd+ijFRmUtQkWSdRV9Wf3ij77W9VPWUOnD3EI51uINUy76x3DET1vdGyM6K0XoFkUI2EMBU/EEIQcc7foMBDMhwW1Rmt75fCRe16DSBw8CgjksdYj17edRnFNCJ0Ph5rdVYgCxqdF36voliIrp2NeptkcTeIeZEEXwTx+ec81VBozn8l836fe4MIQS+bNrh3CJNrcCSqr0FrKOJ3tsZVWImo0UptUiPnISsnk6VDlkNeZDwLnjmIWN08iVH4yFj2q0SWuSnUoDAmPN2AfFyOwGLNR4PS164fp53CJYFwP1j1MkwzSt2lyw2VMXGIuJHTJ9WgquvkZoDDDe5VTrUXydkHO/q+RCAsqLFsP26BfbFusP+7U0V9YznZYciU7nuHH+iCTLocJZdSwLIJIuBSVur0SnYhTJ3KvtFW1hRwRWMnMqrA/qK6uLio8lk105iry8vLCvN5N1kejkxPP58XWSUBVDEEUKH96h9QSwMEFAAAAAgAAAAhXL1vf+diBgAA/w0AABEAAABwcm9tcHRzL3JlYWRlci5tZHVXXW/UOhB9318xggfgKs2FAi99Q6VAdasW0aIrnrrexJu11rGD7XRZ6f74e2bsZLdFSIgmju2ZOefMx/7wI6mgKW00rU2IiWJSnSa/JkVJq76mH9gStGplT9CDjyb5sCflWtoFkzR28nJINV3rX6nCRQ86fwqRtGo21Aa1TuSdJv1LNYl0axKtg+9p78dQjleyIZ+jtQ/5qBv7lQ66JesblYx3ZPAPm9k+XHJ4IDUM1uBUCkZHcVQsxppunN2zEWqwzQ/YvjZWx4qip91GpfzND3tcm/wcY0hkIilrZcUqdkmQibQzWNUPvKAFKN7R+FbXiwVj1as9OZ+o2SgHJJXbp41xXU3fo6YlI3nPLiwJ5g4OTZ9Hd9/4vkdwS8AB3xkIPnQib0PwXVB9PKNlBwg7+EonblnR0kb+PwKo/L7BGf67a5bMC/ub2cJ9vX+Qv622WvhjFypysgtYm6BBkh/TMKaMy7SFQS/bRgeBxBT5otgEMwDuxeL5c3pT07l3KYDoilYaAQgKiNdb0GDt4tskJxPjqOXSrUYkiuKGobcGQizIZsHEfb/ylmkznVNpDEyhDoFtg3TX4fXL3d1XJimNUfiI2d2gsd3Rg7Ij2ANoQf8cEWFkN8E7jPRMFvtbhDe/1XQHF4B4q3lVN1tWlwdTxWp2z+5z5Kc1cWhRUqqYaVkUoSQX6AKJ/YjHFeM+Q1KJZBLSp6YPkwKZr8QC4UQ5gssiTyroEJ6AH/VIVHwZhCMC3QtponCnehyBpHfBu+4MMERY6TidkBhse1WgrspRSSCoSVJ1NRrL+cvwif7XylgWcK9Ss5lzERnlYZDrhuwAxB+9RDZYVeLodY/qUS9O6FargAT3rtESzCNJ07Pbrxfnl58uz+/vbv65uH5GJyf04q962L+g/4jFTSenr5c5bdh077l4Dboxa9OQabVLeAJtXEiOhPIEzczwhvGA9naIQJdymCXHmLV6bRygkIon342UhxHZxoHccBaLD4wG2yjQ5qgO7AColB19c/oaKncQ5MDKYtJeLqHdkO55eSm2ltq1+fVVLsMiHD9Zy+eZXaHk5+iTZnemItR6iNlavzsZB9QqBltLtrJDqCBT3cID3MpBxqPQqyc72MuYSy/ScGU50FVQjjkMWNtoy8FIhlmuw96xOx9KyNp0m/SbViOX8+STsrNWcrNBaZkqY1mPRS9jiqCXln/vfNjGQTV6mdPvLTJ64zk7ETDEKX1oahtxcSc97qg18Xm0DhVKN5F+k9RWXD+crOmrabbHd7ZmvYYMnOi64Z4wq2ZtfgGy0bacUD3qRr6642NSaRCuDygnGV05Lxxyk9hmpKzZaq78XDIQ/VNjvVYOJebIC0nz3aboQhhTTHxmixPARv34xOgaaabC3GG98GnyTRFVQy5/KpbO61JK/iySP8tLSjHrBAl0oFeK/mw1izvtTCOavpKOwB1QFcx2UoYPOcBpgRx9mnI1Xa6Pv8sdzK7cUpVW84jtG7xtsMRMzDNHBJ+J+wOicVkFHfpJnLqMmgaXQ1meD3OYedDJlCKeqSyWPsheTF1aPFKHBMjaflfTv9K+jyYUhZrlcb/KCGzUw/S5E8HfJj9INnGmijzmbQWNxxzJ5KJ52ZfpRaqjy+otAThlEXDccaa7qf2xeiO3zj5PCF5GHuIOEsoEJbp2HM3l7e33izOZGrlb656FV7p97nC4eC6Ur19znraAGT2ML81ZhsIgpspggCMP3j6UIs3z3LR1asO/d+yrm/MPd5c31/TmbPHp8uoii3xQCQ0NFQzcMZ55JpxLTTV5GgFHowtDV5fXF7dnR1M0ezHHXGbYOB3N/FTMW0m046oIVEYLkXy7+AznzhaHIagZgyQpjwnTXceJvILithVHbxDvH0aaQ//73eihP52+z6k1qc6I2pAz7DCblzzDBTZHuMLjNlZ5RTpRnIcCH0wnsjGOu7LkRJ53SvQ6NmqAx6IfUSKvymyU/ZLWjOny/MuH68/gifM14irNo0NU+6zvck4GFumUaIBSl3PfLiKVvD8i/zSTX5FwWFGGXc5kcxy4WmFirgRI9bT614uXIkTP8xHvmK9+W9E7ueg9mUdFiBvBq8XF9Uckw7r8AuGWwZhIe5dxWYVOp1ksba4YUnQmVpZZtqPbOr9zeWh4hA3PINOPnDIBtFX+1QTr9eJ/UEsDBBQAAAAIAAAAIVxuZPNHywAAAFgBAAAPAAAAcHJvbXB0cy9yb290Lm1kZU9BbsMwDLvnFXxAkA/sD7sOPQVeotYCbKmw5XT5fW1l2GUnSRQlkjdtBSrphKkmcIWVIPVOZTVdw4PEFty0IYaDIIo7J3Junce4ac5B9gvB6DpY23dmc+wDFgn+BxIy7XjykxILIWra61jnZfqKweiggky1djbOLlloIz5oBsuW2s7yQOhgZtk704a4GEsjqI+X7DxufzOFzVjFU3V2SOl/OrzY4mVwHQZRaTj/s7lMn26sn716YYHRTxfxqL7xv0FOaM9SPPUyvQFQSwMEFAAAAAgAAAAhXHm3D1pvAgAAcwQAABcAAABzdWJfYWdlbnRzL2FwcGxpZXIueWFtbG1UwY7TMBC95ysG7YGt1GTb0C1sOCAkOCAkxGEP3CI3njRWHTt4JpTVin9nbKfLgri0nsn4zbw3L7mCj9++lpvbGupNvS+323LzugEeMOdfvQLr/QTqiI7hmtSIwN5bWkM6y89kjTtCbyyu4Gx4AAVT8OPEgqIYOhWCQUqQAZXG8JLkMPnAoJxO+d78QNBB9Rzrgp+PAzxOVrl3v1LNY3rWfllCRmsJjDTwwOEhQoxgHPgg6FVxJefOzhrbzjuWuQk09mq2QsB5BucdrjJHNcn0GGCciYEQU5IVnWBEIiEN1xeManpobvd35e2bjYzqpzTA7KRjBqzgE4PJPK2KcBzvx3kHbzUBzYfRcDsp7gZRz4tOOQXo9KLP7OA6Sd3K0QkZaXpXb8q7ul4lKJl/iHIHLKUi3+Igg4s+MonhtwLL+DOBxsKzn60opk6Zm5u1DCUzDP9ptK/L7aberdZwHkw3pAvLKnsfDkZTFPdesu8/fJYusnoVu1KnllGMIw5zx8bLNlwn8j3LUDuzsUnIrXRJfLQXa8SlCKEIk1BEFmLDM6POrkiMkmaHoDq5IahGC+nlsQrRiT1WhRNTNpe1FhqpC2aKzRu4f7JhuiTYqA3Tk3GyeWOujWZeQzdgdyJAJVJEMms4IU60ODbQYvDcjBKdSRGhvBvxnJeby9PSq2L0Gm0DRxxHVe5K3B1Kw+V3xeV5p7b7suPimV4NvFiMDFV1kxdBNwu3atTFvzZvLj4v0ivaFABl9JQUSEOncyzvYCKYoie6KToHw/gnPCK34mKeKYXPDVyIdzDI9i/N439vjn/PnHN0c/lKVA9qtMVvUEsDBBQAAAAIAAAAIVyjEUSxowEAAE4DAAAVAAAAc3ViX2FnZW50cy9kcmFmdC55YW1sjZJBb9NAEIXv+RVP6oFEso2TJq1whRCHckFCPXKzBnscr7TZtXbWDQjx35ldh1IJDrms7Nnnmfe+8Q0evz6V9WGHXb27K7fbsr5vMJhnRh9oiBwExmGiQNayxfoz3uNQ4GziqPU4MvaIHnfwQ37rjUwUu3FT4ZG6ESNpgyjwZ4ch0Mm4Iyhm6RC8i+m7dG+cxDB30XhXQDycxxR4MNbCCGSkwH0Bcv1Lt8iniQPFOTDWdXVboK4O6bhPx7sC26p+QDd6YW3pvA6dHfebanWzWDsHE1l7OQZ/py6CexNTGGER9QGJFBmz6zksNNovD0vIP2iEORcCk4reiD5MPkSsU/HnZMl9+JWNWu20yfbVyJggsBX1bVxn557bTlGw02RO7WhOyW3zbvY1nn2aps4dnbhZxq/oqB+0nSWRBk+XBX1MxVXP0gUzJZgNPqVlGg0xsR4uJfS2tJpxIRCkAGceutMXuuSOlrPhV5wvwjz/H3BJe0H6H4zVSuZvbfYszQoooZEHc2z1bxkb/L18e4HbbqsfdLJXSXfXS2+vl+6vlx4W6W9QSwMEFAAAAAgAAAAhXAm/KBnGAQAAJQMAABkAAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMS55YW1shVLBjtMwEL3nKx7aw4LUZJNs2xW+ICR6XXFbJIQiN5kmVh072BNahPh3bG9aVsuBU2YmM2/e+L0b7L58zstNjbqst3lV5eWDQOfkgcmhgj1gs4I0vSb4UWpNntEOoUArMI0TOcmzI5TFfQG2Vnt8/RYGOijT6rmjprWGybCHsYYEFMMTefBAcCQ7crc+BJN1HGrOzv2AX5OW5sPvBGMsD8r0IO2pyG4wynNjZ55mbtgeyXjclw91ao2Nx9Db7OeuJ0ZV1mu85SFihv10ToQ6HJwd0/505zsB+RxB+RBquSeNSc8eVnfPHOgUjj3zKmUS7czLhLY+3KIC+uPuCXtt2+MK3ib0Vk4RMRyQ0h82PKm/Rb3ZFvi4zJ8UD6EDbQCKV+4eP0ErE4btHN9M+risyIwc6apLU2Ud+dapiZU1Ak9OvVTrvzplo+1IC/Q0jjJf57Te54rz75Lz01pW27zlTBnPbm6fF7xZtERR3E3h8Sb2d1cuxdhlr7UWSexs0elIPxfugXnyiAgmyXoykdV1Kn4PqhcZXjIWkXIs2amZYvJ+s2RHge06xP84QiRLxK6LIf4i47VLRLJJ+nO54uIYgYMMrsv+AFBLAwQUAAAACAAAACFc5s4lwR0BAADhAQAAGQAAAHN1Yl9hZ2VudHMvZHJhZnRlcl8yLnlhbWx1j8tOwzAQRff+ikFs6zQJaat6zx6JBUgIRW48Sa3ED+yxCn9PXNooG1b2vO499xGe3194uauhLus9rypeHgSoIHvCADW4HnYbkHaYEDqZIm6A0HgMklJAKItdAa/SIJBzU9zARVvlLvOBmtd9BBkXsapgdt5c1NuaKYxd0J60swLegl57/uPGjFM4CRjQGMkbjs2Ja+JfkvilkdWed8S0jRRS9yf7oG03JYVQFFsfnPEUtwtBYRS7zdvOWUJLUYB1FplL5BO1I/7ciGfea0gBH59sQJuplqv89noQDNbEIiPnlvOtz8XxXo0C9s38N/K7vTmRG9HO6k/loc5bZ21HbYeV8qp5SmpAElCVdXOd3FPQ2aXhnFP0corIfgFQSwMEFAAAAAgAAAAhXBcAT9MdAQAA4QEAABkAAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMy55YW1sdY/LTsMwEEX3/opBbOs0SdNWeM8eiQVICEUmnrhW4gf2WIW/JykhyoaVPa97z72Hx9cnXh5rqMv6xKuKl2cBKsqeMMIBfA/HHUinRwSdZVQ7ILQBo6QcEcriXMCztAjk/Zh2cDVO+et0oKCTIYFMq1hVMDdtrurtgSlMXTSBjHcCXqLZev7jxqxXOArQaK3kDcfmgxvin5L4tZHViXfEjEsUc/cre2dcN2aFUBT7EL0NlPYrQWEVW+Zt5x2hoyTAeYfMZwqZ2gG/F+KJ9xZSwNs70+hmqvVqfnujBYMtsZiR55YPbZiLh+NSDQJOzfS38qtdnMgP6Cb1Q3mu562LcYNxeqO8aX5kpZEEVGXd3CZ/Kejis77MKXo5JmQ/UEsDBBQAAAAIAAAAIVzBO3NCJQEAAAMCAAAZAAAAc3ViX2FnZW50cy9kcmFmdGVyXzQueWFtbI2PS0+EMBSF9/0V17idMoDMTOzevYkLTYwhHXrpNNCH7SXovxcQCUtXva+e8517eHp75vmphDIvz7woeH4RoKJsCSNU4Fs4HUA63SM03lGUDcHoo4LWx6U4AKENGCUNESHPHjN4kRaBvO/TAUbjlB8nBQWNDAlk2tSLjLnpcrOrK6YwNdEEMt4JeI1mD/Ffe2a9wl6ARmslrzhWV26If0riYyWLM2+IGZcoDs2vz51xTT8ohCw7huhtoHTckDKr2LqvZwJ0lAQ475D5gcJAdYffa4QpwJJawPsH0+hmqu3X/LZGCwZ7YjEjzyMf6rA0p7XrBJyrqbbyq16dyHfoJvWH/FLOVzfjOuP0Tnk3vA5KIwko8rJaNn8p6OYHfZtTtLJPyH4AUEsDBBQAAAAIAAAAIVx8rCm2JwEAAPkBAAAZAAAAc3ViX2FnZW50cy9kcmFmdGVyXzUueWFtbIWPy07DMBBF9/6KQWzrNAlJq3rPHokFSAhFbjxxrMQP7IkKf09SQlRWrDyexz333sPj6xPP6xLKvDzwouD5UYCKsiOMUIPvoN6BdHpEkKBM12FER5B6GXAHhDZglDRFhCLLM3iWFoG8H9MOLsYpf5mPFbQyJJBpEy4y5ubNjdTUTGFqowlkvBPwEs0f/v9kZr3CUYBGayWvOFZnboh/SOKXShYH3hIzLlGc2h/EnXHtOCmELNuH6G2gtN/cZFaxdd603tEMTgKcd8j8RGGiZsCv1f3s/RpYwNs70+gWV9vV8nZGCwa3jsVieWn50AQBeXaq198g4FDNtZWfzUoiP6Cb1R/yY7ls9cYNxukb5ZvmeVIaadbPy+o6+U1BvZ90v6To5JiQfQNQSwMEFAAAAAgAAAAhXDUMDudVAQAAXwIAABgAAABzdWJfYWdlbnRzL3BpcGVsaW5lLnlhbWyFkT9PwzAQxfd8iiexgNSUthIIZUBiYGNAwMBWHc6lsXDs4Lu25NtjOwU22Oz7897Pz2e4f32sV9drbFab63q9rlc3DbTnuX61wcsGMQTFEA7coufI2HvTk9+lK38aHhVWBZ4GxrkceUs79or6FqMd2VnPFwtIAHk8uOGuNIuiSZWk0yJ0HTQklSVekrMJvrM7jKS9QJQmRHak9sB5KrNpZC4ayyrbNj9OVfHeGkciDZ75Y5+ullxxrVoWE+2oNvgGT0xtEYs8BrEa4oSj1R6UKtRyTE1S9MG1UirJPDhB8G7C2wTrRePeZLEFHCu6GTC42rEIjtEqRwGT6dFG6jStckqMjIJbmxZiGEp0mSDqItN40DgmgwxWliQZIcTMU+iyRh6x34CmZ/N+sslhyv5tSKLLKh3mr5CmAupTrNsca4Pf5uX82uVEg/tzruD8P3aCmwe/AFBLAwQUAAAACAAAACFcppUYW+UCAAAzBQAAFgAAAHN1Yl9hZ2VudHMvcmVhZGVyLnlhbWxtVMFuEzEQve9XDO2BVspuk5KGskggVKWlUtUi4MAtcnZns1a8trHHTSPEvzO2k6YgTrse2zPvzXvjY5j/+FKOL87hfHw+KyeTcvy2BoeiBTJGeTBabWG5Bak9udCQNLoCF/SiMcMgdAvSg4DOIYLvUSk48RtcIe8BPmET0gW7radvT0fgDRxpA9hKyumPoDeq9bEA9QjWmcESxLTa0D4aT4KSnt6nZQQn9cpD02Ozfg6he82hHSgtBvRVccywGxVaZLSaUJOHFjsRFNUv7kUKnXSeEkDJhzZSt2YT46m+8GvgfF6sEFByyMFGbEcJp9hn3F9aI9r/3JNdjsVGcV585CQOS+4knCQJpmO4nkxAGU+MAYKOyLTReBp5mEA20GKNW7BK6Ay/k1ooIHwiOKHehFVPYIVjAviUaLcjUGpYcH1NUYTLi3fl5ezNaSTO3cswN04SOp/YxLWwVkmuTb2LKVPsVyz68TfXGviPcASbXjZ9ZOJwGSTT71g78MSb7JnEb8t7PwP6LCi31gf3KB8xyWRF9hJzS/Rn04MD372k59AaR9yJDTTCOcnXg2VP8PYjgg7DEh22cPdw9en77cM9LJVp1h5Orm/v5iO4u72ffxvB1/kN743g6vOn+5v56XtonegoSp//mD0dUsgEuBOK/b8U0WPmsDmp4DuDy0yjHvu+mWQMj0TJncIhi9j0Qq+wreBKWDBdctyikwqZTEw/g/IDXLJNX/h/D5ePWJiN44nzCx4AzdQtl2CCIvcu2rzeubho0TdO2rhTw4NFneVtTMu265nfEFiKDCiBzp3Np6T3gVXNbVe7P3xikaAJzrF/+PiKUz/TlXGg2GK7jFUxcCFVQ5r9clridFlKKn8KKjdTMZmVDRUvHpEaXu1mE6rqLBP3Z5lKNbTFv4Nb7+esOIxCnWahSC9JXQCUh/bm1eGZSusV0iLqFnzBE4GOFdznj99Orv5GlWP+zAt2PYtabcWgij9QSwECFAAUAAAACAAAACFcVjaEnpsBAADdAgAACgAAAAAAAAAAAAAApAEAAAAAYWdlbnQueWFtbFBLAQIUABQAAAAIAAAAIVzY/KEY1QAAABgBAAAVAAAAAAAAAAAAAACkAcMBAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAACFc6A3n1sQBAAAZAwAAEAAAAAAAAAAAAAAApAHLAgAAZXZhbF9jb25maWcueWFtbFBLAQIUABQAAAAIAAAAIVxq31bzJAcAAG8PAAASAAAAAAAAAAAAAACkAb0EAABwcm9tcHRzL2FwcGxpZXIubWRQSwECFAAUAAAACAAAACFcg50brdYCAADGBQAAFAAAAAAAAAAAAAAApAERDAAAcHJvbXB0cy9kcmFmdGVyXzEubWRQSwECFAAUAAAACAAAACFc90Lm3zgDAACcBgAAFAAAAAAAAAAAAAAApAEZDwAAcHJvbXB0cy9kcmFmdGVyXzIubWRQSwECFAAUAAAACAAAACFckYnf7woDAAA2BgAAFAAAAAAAAAAAAAAApAGDEgAAcHJvbXB0cy9kcmFmdGVyXzMubWRQSwECFAAUAAAACAAAACFcQbufeScDAAB4BgAAFAAAAAAAAAAAAAAApAG/FQAAcHJvbXB0cy9kcmFmdGVyXzQubWRQSwECFAAUAAAACAAAACFc+DbFOCwDAACVBgAAFAAAAAAAAAAAAAAApAEYGQAAcHJvbXB0cy9kcmFmdGVyXzUubWRQSwECFAAUAAAACAAAACFcvW9/52IGAAD/DQAAEQAAAAAAAAAAAAAApAF2HAAAcHJvbXB0cy9yZWFkZXIubWRQSwECFAAUAAAACAAAACFcbmTzR8sAAABYAQAADwAAAAAAAAAAAAAApAEHIwAAcHJvbXB0cy9yb290Lm1kUEsBAhQAFAAAAAgAAAAhXHm3D1pvAgAAcwQAABcAAAAAAAAAAAAAAKQB/yMAAHN1Yl9hZ2VudHMvYXBwbGllci55YW1sUEsBAhQAFAAAAAgAAAAhXKMRRLGjAQAATgMAABUAAAAAAAAAAAAAAKQBoyYAAHN1Yl9hZ2VudHMvZHJhZnQueWFtbFBLAQIUABQAAAAIAAAAIVwJvygZxgEAACUDAAAZAAAAAAAAAAAAAACkAXkoAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMS55YW1sUEsBAhQAFAAAAAgAAAAhXObOJcEdAQAA4QEAABkAAAAAAAAAAAAAAKQBdioAAHN1Yl9hZ2VudHMvZHJhZnRlcl8yLnlhbWxQSwECFAAUAAAACAAAACFcFwBP0x0BAADhAQAAGQAAAAAAAAAAAAAApAHKKwAAc3ViX2FnZW50cy9kcmFmdGVyXzMueWFtbFBLAQIUABQAAAAIAAAAIVzBO3NCJQEAAAMCAAAZAAAAAAAAAAAAAACkAR4tAABzdWJfYWdlbnRzL2RyYWZ0ZXJfNC55YW1sUEsBAhQAFAAAAAgAAAAhXHysKbYnAQAA+QEAABkAAAAAAAAAAAAAAKQBei4AAHN1Yl9hZ2VudHMvZHJhZnRlcl81LnlhbWxQSwECFAAUAAAACAAAACFcNQwO51UBAABfAgAAGAAAAAAAAAAAAAAApAHYLwAAc3ViX2FnZW50cy9waXBlbGluZS55YW1sUEsBAhQAFAAAAAgAAAAhXKaVGFvlAgAAMwUAABYAAAAAAAAAAAAAAKQBYzEAAHN1Yl9hZ2VudHMvcmVhZGVyLnlhbWxQSwUGAAAAABQAFAA0BQAAfDQAAAAA"               # filled in by tools/build_notebook.py --embed
# Model: taken from the embedded zip's agent.yaml. Set MODEL_NAME only as a cross-check (the run asserts the
# zip declares exactly this name), e.g. --set MODEL_NAME='"gemma-4-e2b-it-qat-w4a16-ct"'. The kernel-metadata
# model_sources must attach google/gemma-4/other/<name>/2 for the same name.
MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"   # overridden by --set

# Compaction settings: Q7 ANSWERED [V] from HARNESS_README.md section 7.2 (interval 15, overlap 2,
# token_threshold 14,336, event_retention_size 5). The 'interval 5' reading confused retention with interval.
COMPACTION_INTERVAL = 15   # [V] HARNESS_README §7.2: interval 15, overlap 2, threshold 14,336, retention 5
COMPACTION_TOKEN_THRESHOLD = 14336

In [ ]:
# ---------------------------------------------------------------- environment + wheelhouse (hosts' recipe)
# Adapted from the organizers' Getting-Started notebook as mirrored by community repos.  # VERIFY
import glob, importlib, json, os, shutil, subprocess, sys, time, zipfile, io, base64
from pathlib import Path

os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1', 'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
WHEELHOUSE_DIR = next(Path(p) for p in ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
                                        '/kaggle/input/gemma-4-developer-agent-wheelhouse'] if Path(p).exists())
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
# The wheelhouse ships cp312 binary wheels (orchestrator env = Python 3.12), but the grading sandbox and
# Kaggle's CPU image are Python 3.13 (docker/Dockerfile.sandbox is `python:3.13-slim`). A single bulk
# `pip install ... check=True` aborts the WHOLE install when one wheel is incompatible, which is what killed
# the first EXP-000 push (apache_tvm_ffi cp312 on py3.13). Filter by interpreter tag, then assert that the
# packages we actually need for this run arrived.
# Import names, not distribution names: the `google_adk` wheel installs the `google.adk`
# namespace package, so importing 'google_adk' always fails. [V] google_adk-1.36.1.dist-info
REQUIRED_PKGS = ('swegemma', 'adk_submission', 'adk_eval_core', 'google.adk')

def _wheel_is_compatible(path: Path) -> bool:
    """True when the wheel's tags intersect this interpreter's supported tags."""
    try:
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
    except Exception:  # noqa: BLE001 - fall back to accepting everything
        return True
    try:
        _, _, _, tags = parse_wheel_filename(path.name)
    except Exception:  # noqa: BLE001 - odd filename: let pip decide
        return True
    supported = set(sys_tags())
    return any(t in supported for t in tags)

all_wheels = sorted(tmp_whl.glob('*.whl'))
usable = [w for w in all_wheels if _wheel_is_compatible(w)]
skipped = [w.name for w in all_wheels if w not in usable]
T_INSTALL = time.time()
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                       *(str(w) for w in usable)], capture_output=True, text=True)
if proc.returncode != 0:
    print('bulk install failed; retrying wheel-by-wheel to isolate the bad ones')
    failed = []
    for w in usable:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            failed.append(w.name)
    skipped += failed
importlib.invalidate_caches()
print(f'wheelhouse: installed {len(all_wheels) - len(skipped)}/{len(all_wheels)} wheels in {time.time() - T_INSTALL:.0f}s '
      f'(python {sys.version_info.major}.{sys.version_info.minor})')
if skipped:
    print(f'  incompatible/failed ({len(skipped)}):', ', '.join(sorted(skipped)[:12]))
missing = []
for _pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(_pkg)
    except Exception as _exc:  # noqa: BLE001
        missing.append(f'{_pkg} ({_exc.__class__.__name__}: {_exc})')
assert not missing, 'required packages unavailable: ' + '; '.join(missing)
print('  required packages present:', ', '.join(REQUIRED_PKGS))

DATA_DIR = next(Path(p) for p in ['/kaggle/input/competitions/gemma-4-developer-agent',
                                  '/kaggle/input/gemma-4-developer-agent'] if Path(p).exists())
RUN_DIR = Path('/kaggle/working/runs') / EXP_ID
for sub in ('traces', 'patches', 'logs', 'phase0'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, ' run dir:', RUN_DIR)

In [ ]:
# ---------------------------------------------------------------- Phase 0 checks (plan §3.4)
# Answers Q1/Q2/Q3/Q7/Q9 by reading the installed harness source; saves HARNESS_README.md and
# sample_submission/ to the run dir so they can be downloaded and read offline.
if RUN_PHASE0_CHECKS:
    import importlib.metadata as md, re
    report = {'python': sys.version, 'packages': {}}
    for pkg in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm'):
        try:
            report['packages'][pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            report['packages'][pkg] = None
    for name in ('HARNESS_README.md',):
        src = DATA_DIR / name
        if src.exists():
            shutil.copy(src, RUN_DIR / 'phase0' / name)
    ss = DATA_DIR / 'sample_submission'
    if ss.exists():
        shutil.copytree(ss, RUN_DIR / 'phase0' / 'sample_submission', dirs_exist_ok=True)
        print('--- sample_submission files (Q2) ---')
        for p in sorted(ss.rglob('*')):
            if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.json'):
                print(f'===== {p.relative_to(ss)}'); print(p.read_text()[:4000])

    def grep_pkg(pkg: str, patterns: list[str]) -> dict:
        hits = {}
        try:
            root = Path(importlib.import_module(pkg).__file__).parent
        except Exception as exc:  # noqa: BLE001
            return {'error': str(exc)}
        for pat in patterns:
            rx = re.compile(pat)
            for f in root.rglob('*.py'):
                for i, line in enumerate(f.read_text(errors='ignore').splitlines(), 1):
                    if rx.search(line):
                        hits.setdefault(pat, []).append(f'{f.relative_to(root)}:{i}: {line.strip()[:160]}')
        return hits

    report['Q1_tool_registry'] = grep_pkg('swegemma', [r'register\(', r'def (run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph)\b'])
    report['Q1_skill_tools'] = grep_pkg('adk_submission', [r'run_skill_script|load_skill_resource|SkillToolset'])
    report['Q2_eval_config_keys'] = grep_pkg('swegemma', [r'max_time_minutes|max_tool_calls|max_turns|timeout_seconds', r'load_submission_eval_config'])
    report['Q2_agent_tool_schema'] = grep_pkg('adk_submission', [r'agent_tool|config_path|skip_summarization'])
    report['Q3_thinking_forwarding'] = grep_pkg('adk_submission', [r'thinking_budget|thinking_token_budget|enable_thinking|include_thoughts'])
    report['Q7_compaction'] = grep_pkg('swegemma', [r'compaction_interval|token_threshold|EventsCompactionConfig'])
    report['Q9_extensions_limits'] = grep_pkg('swegemma', [r'ALLOWED_SUBMISSION_EXTENSIONS|MAX_SUBMISSION_SIZE_BYTES|allowed_file_extensions|max_total_size_bytes'])
    report['Q2_grading_timeout'] = grep_pkg('swegemma', [r'timeout', ]) if False else grep_pkg('swegemma', [r'verification.*timeout|timeout.*verif|command_timeout_seconds'])
    (RUN_DIR / 'phase0' / 'phase0_report.json').write_text(json.dumps(report, indent=1))
    print(json.dumps({k: (v if k == 'packages' else {kk: len(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
                      for k, v in report.items()}, indent=1))
    print('full grep hits saved to', RUN_DIR / 'phase0' / 'phase0_report.json')

In [ ]:
# ---------------------------------------------------------------- tasks + candidates
from swegemma.models import load_tasks  # [V] swegemma/models/__init__.py exports load_tasks(path)->list[Task]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
from collections import Counter
print('repo counts:', Counter(t.repo for t in all_tasks))
if TASK_IDS:
    want = set(TASK_IDS)
    candidates = [t for t in all_tasks if t.instance_id in want]
else:
    pool = [t for t in all_tasks if (not REPO_FILTER or t.repo == REPO_FILTER) and t.instance_id not in set(EXCLUDE_IDS)]
    candidates = sorted(pool, key=lambda t: t.instance_id)[:N_TASKS]
print(f'{len(all_tasks)} tasks; candidates ({len(candidates)}):', [t.instance_id for t in candidates])

In [ ]:
# ---------------------------------------------------------------- unpack our submission
AGENT_DIR = Path('/kaggle/working/agent')
shutil.rmtree(AGENT_DIR, ignore_errors=True); AGENT_DIR.mkdir(parents=True)
if SUBMISSION_B64:
    zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)
elif Path(SUBMISSION_ZIP).exists():
    zipfile.ZipFile(SUBMISSION_ZIP).extractall(AGENT_DIR)
elif not RUN_AGENT and (DATA_DIR / 'sample_submission').is_dir():
    # A gold-control run never calls a model, so it needs no submission of ours. Fall back to the
    # official sample_submission: it is a real, schema-valid submission dir, so the model/adapter/limits
    # validation below still runs for real instead of being skipped.
    shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR, dirs_exist_ok=True)
    print('no submission supplied; using official sample_submission (RUN_AGENT=False)')
else:
    raise SystemExit('no submission: set SUBMISSION_ZIP to your uploaded dataset file or embed with tools/build_notebook.py')
print('agent files:', sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))
import yaml
EVAL = (yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) or {}).get('evaluation', {}) \
    if (AGENT_DIR / 'eval_config.yaml').exists() else {}
if not RUN_AGENT:
    # The official sample eval_config is deliberately tiny (1 min / 10 tool calls) to demo the schema.
    # Gold control runs no agent loop, so those caps only risk failing a sound task for budget reasons
    # and corrupting the Q5 answer. Give the grader room; the agent budget is irrelevant here.
    EVAL = {**EVAL, 'timeout_seconds': 300, 'max_time_minutes': 60.0, 'max_tool_calls': 100}
print('eval_config:', EVAL)

In [ ]:
# ---------------------------------------------------------------- vLLM server (hosts' recipe)  # VERIFY
# CPU-safe imports only: a gold-control run (RUN_AGENT=False) never loads vLLM/litellm/torch, and on
# Kaggle's Python 3.13 CPU image those cp312 wheels are not installed at all.
from adk_submission import ModelRegistry, discover_adapters, discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits

# D-004 2026-11-05: TARGET was hard-coded to 31B; it now comes from the zip, so a small-variant candidate
# runs without editing this file. Mirror of tools/submission_rules.py ALLOWED_MODEL_NAMES (the notebook runs
# on Kaggle and cannot import tools/). 12B is excluded there and here: it needs vLLM --hf-overrides
# (HARNESS_README §3.1 MODEL_VLLM_EXTRA_ARGS), which this cell does not pass.
ALLOWED_MODELS = ('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct')
# [V] adk_submission.discovery.discover_declared_models(submission_dir) -> set[str]; walks the root
# agent, sub_agents and agent_tools. There is no `validate_single_declared_model` anywhere in the
# wheelhouse -- that was an invented name. Exactly one model may be declared, so assert the set size.
declared = discover_declared_models(str(AGENT_DIR))
assert len(declared) == 1, f'submission must declare exactly one model, got {sorted(declared)}'
TARGET = next(iter(declared)).strip()
for _prefix in ('openai/', 'google/', 'hosted_vllm/', 'custom/'):   # [V] HARNESS_README §3.2 normalize_model_name
    if TARGET.startswith(_prefix):
        TARGET = TARGET[len(_prefix):]
        break
assert TARGET in ALLOWED_MODELS, f'declared model {TARGET!r} not in {ALLOWED_MODELS}'
assert not MODEL_NAME or MODEL_NAME == TARGET, f'MODEL_NAME={MODEL_NAME!r} but the zip declares {TARGET!r}'
MODEL_DIR = f'/kaggle/input/models/google/gemma-4/other/{TARGET}/2'   # [V] path layout EXP-006 resolved 'exact'
print('model:', TARGET, '->', MODEL_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
# EXP-000 gold control needs no model: it applies each task's reference patch and runs test_patch.
# Gate the server so a gold-control-only run costs ZERO GPU quota (plan §5.7) and can use a CPU notebook.
NEED_SERVER = RUN_AGENT or MEASURE_DECODE_SPEED
server = None
SERVER_START_S = None
target_id = None
models = ModelRegistry()

if NEED_SERVER:
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    from google.adk.models.lite_llm import LiteLlm
    try:
        from adk_submission.resolvers.generation import install_litellm_reasoning_patch
        install_litellm_reasoning_patch()
    except Exception as exc:  # noqa: BLE001
        print('reasoning patch not installed:', exc)
    litellm.drop_params = True
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    assert gpu_count >= 1, 'no GPU: select the L4 x4 accelerator'
    assert Path(MODEL_DIR, 'config.json').exists(), (
        f'{MODEL_DIR} missing: kernel-metadata.json model_sources must list google/gemma-4/other/{TARGET}/2')
    tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    t0 = time.time()
    server = VllmServer(VllmConfig(
        model=MODEL_DIR, port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        default_chat_template_kwargs={'enable_thinking': True}, max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
        tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
    server.start()
    SERVER_START_S = round(time.time() - t0, 1)
    print(f'vLLM up in {SERVER_START_S}s (tp={tp}, gpus={gpu_count}) at {server.base_url}')
    target_id = getattr(server, 'effective_model_path', MODEL_DIR)
    for alias in {next(iter(declared)), TARGET}:
        models.register(alias, LiteLlm(model=f'openai/{target_id}', api_base=server.base_url, api_key='EMPTY'))
else:
    print('vLLM server SKIPPED (RUN_AGENT and MEASURE_DECODE_SPEED are both False) - no GPU required.')

In [ ]:
# ---------------------------------------------------------------- Q10: decode speed probe
DECODE = None
if MEASURE_DECODE_SPEED:
    import urllib.request
    body = json.dumps({'model': target_id, 'messages': [{'role': 'user', 'content': 'Count from 1 to 400 separated by spaces.'}],
                       'max_tokens': 400, 'temperature': 0.0, 'chat_template_kwargs': {'enable_thinking': False}}).encode()
    req = urllib.request.Request(server.base_url.rstrip('/') + '/chat/completions', data=body,
                                 headers={'Content-Type': 'application/json'}, method='POST')
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as resp:
        out = json.loads(resp.read())
    dt = time.time() - t0
    ct = out.get('usage', {}).get('completion_tokens', 0)
    DECODE = {'completion_tokens': ct, 'seconds': round(dt, 2), 'tok_s': round(ct / dt, 1) if dt else None}
    print('decode probe:', DECODE)

In [ ]:
# ---------------------------------------------------------------- evaluator helpers
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig  # [V] google/adk/apps/_configs.py:50
from swegemma.evaluate import Evaluator

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def make_config(results_dir: Path) -> EvalConfig:
    limits, gen = build_submission_limits()
    return EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
        timeout_seconds=int(EVAL.get('timeout_seconds', 300)),
        max_time_minutes=float(EVAL.get('max_time_minutes', 60.0)),
        max_tool_calls=int(EVAL.get('max_tool_calls', 100)),
        max_turns=int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=COMPACTION_INTERVAL, overlap_size=2,
                                                        token_threshold=COMPACTION_TOKEN_THRESHOLD, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False)

def record(r, task, t_start: float, t_end: float) -> dict:
    patch = r.agent_patch or ''
    dur = float(r.duration_seconds or 0)
    wall = t_end - t_start
    return {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
            'status': getattr(r, 'status', None), 'test_exit_code': r.test_exit_code,
            'duration_s': round(dur, 1), 'task_wall_s': round(wall, 1),
            'setup_s': round(max(0.0, wall - dur), 1),   # everything outside the agent loop (setup + grading)  VERIFY
            'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
            'total_tokens': getattr(r, 'total_tokens', None), 'error': getattr(r, 'error', None) or getattr(r, 'error_message', None),
            'patch_chars': len(patch), 'patch': patch, 'trace': str(getattr(r, 'trace_path', '') or getattr(r, 'trace_json_path', '') or ''),
            'test_tail': (r.test_output or '')[-1500:], 'max_time_minutes': EVAL.get('max_time_minutes'),
            'max_tool_calls': EVAL.get('max_tool_calls')}

def append_row(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as fh:
        fh.write(json.dumps(row) + '\n')

In [ ]:
# ---------------------------------------------------------------- EXP-000 gold control (Q5)
# `sound` must exist even when the gold control is skipped, or the agent cell below raises NameError.
sound = list(candidates)
if RUN_GOLD_CONTROL:
    from adk_eval_core.tracing import SessionTrace  # [V] tracing/__init__.py re-exports it; no-arg ctor

    class GoldEvaluator(Evaluator):
        """Phase 1 returns the reference patch; no model call, nothing is shown to any agent."""
        async def _run_agent_sandbox(self, task, *args, **kwargs):  # [V] evaluate.py:194 -> tuple[str, str|None, SessionTrace]
            return task.patch, None, SessionTrace()

    gold = GoldEvaluator(make_config(RUN_DIR / 'gold_control'))
    gold_rows = []
    for i, t in enumerate(candidates, 1):
        t0 = time.time()
        r = run_sync(lambda: gold.evaluate_task(task=t, task_index=i, total_tasks=len(candidates)))
        row = {'instance_id': t.instance_id, 'repo': t.repo, 'mode': 'gold', 'resolved': bool(r.resolved),
               'exit': r.test_exit_code, 'status': getattr(r, 'status', None), 'seconds': round(time.time() - t0, 1), 'error': (getattr(r, 'error', '') or '')[:300],
               'tail': (r.test_output or '')[-600:]}
        gold_rows.append(row)
        append_row(RUN_DIR / 'gold_control.jsonl', row)
        print('gold', t.instance_id, 'resolved=', row['resolved'], f"{row['seconds']}s")
    failures = [x['instance_id'] for x in gold_rows if not x['resolved']]
    (RUN_DIR / 'gold_failures.txt').write_text('# gold-control failures (exclude from cohorts)\n' + '\n'.join(failures) + '\n')
    sound = [t for t, x in zip(candidates, gold_rows) if x['resolved']]
    print(f'gold control: {len(sound)}/{len(candidates)} sound; failures: {failures}')

In [ ]:
# ---------------------------------------------------------------- baseline agent eval
summary = {'exp_id': EXP_ID, 'server_start_s': SERVER_START_S, 'decode_probe': DECODE, 'eval_config': EVAL,
           'compaction': {'interval': COMPACTION_INTERVAL, 'token_threshold': COMPACTION_TOKEN_THRESHOLD},
           'tasks': [], 'resolved': 0}
if RUN_AGENT:
    ev = Evaluator(make_config(RUN_DIR / 'official'))
    for i, t in enumerate(sound, 1):
        t0 = time.time()
        r = run_sync(lambda: ev.evaluate_task(task=t, task_index=i, total_tasks=len(sound)))
        row = record(r, t, t0, time.time())
        append_row(RUN_DIR / 'task_results.jsonl', row)
        (RUN_DIR / 'patches' / f'{t.instance_id}.patch').write_text(row['patch'])
        for cand in (Path(row['trace']) if row['trace'] else None, RUN_DIR / 'official' / 'traces' / f'trace_{t.instance_id}.json'):
            if cand and cand.exists():
                shutil.copy(cand, RUN_DIR / 'traces' / f'trace_{t.instance_id}.json'); break
        summary['tasks'].append(t.instance_id); summary['resolved'] += int(row['resolved'])
        print(f"[{i}/{len(sound)}] {t.instance_id} resolved={row['resolved']} calls={row['tool_calls']} "
              f"agent={row['duration_s']}s setup={row['setup_s']}s err={(row['error'] or '')[:100]}")
    n = len(summary['tasks'])
    print(f"RESOLVED {summary['resolved']}/{n}" if n else 'no tasks run')
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))

In [ ]:
# ---------------------------------------------------------------- bundle for download + stop server
# Official logs may contain repository snippets: keep the bundle private, never commit it (plan §1.3).
for extra in (RUN_DIR / 'official' / 'logs',):
    if extra.exists():
        shutil.copytree(extra, RUN_DIR / 'logs', dirs_exist_ok=True)
shutil.make_archive(str(Path('/kaggle/working/runs') / EXP_ID), 'zip', RUN_DIR)
print('bundle:', Path('/kaggle/working/runs') / f'{EXP_ID}.zip')
try:
    server.stop()
except Exception:  # noqa: BLE001
    pass